# Tüketim ve Üretim Tahmini - Weighted Moving Average (WMA) Baseline
Bu notebook'ta EPİAŞ'ın kendi tahminleri (KGÜP/LEP) ile basit bir WMA modelinin performansını karşılaştırıyoruz.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from db.connection import get_db_engine
engine = get_db_engine()

In [ ]:
# Fetch Data (Last 6 Months)
query = '''
    SELECT 
        c.ts,
        c.consumption_mw as actual_consumption,
        l.load_forecast_mw as epias_load_forecast,
        g.wind_mw as actual_wind,
        k.wind_mw as epias_kgup_wind,
        g.solar_mw as actual_solar,
        k.solar_mw as epias_kgup_solar,
        g.total_mw as actual_gen_total,
        k.total_mw as epias_kgup_total
    FROM raw_actual_consumption_hourly c
    LEFT JOIN raw_load_forecast_hourly l ON c.ts = l.ts
    LEFT JOIN raw_actual_generation_hourly g ON c.ts = g.ts
    LEFT JOIN raw_kgup_hourly k ON c.ts = k.ts
    WHERE c.ts >= CURRENT_DATE - INTERVAL '6 months'
    ORDER BY c.ts ASC
'''
df = pd.read_sql(query, engine)
df['ts'] = pd.to_datetime(df['ts']).dt.tz_convert('Europe/Istanbul')
df = df.set_index('ts')
df.head()

In [ ]:
# Baseline 1: Weighted Moving Average for the SAME HOUR of the day
# Example: Predict today's 14:00 using weighted average of the last 7 days' 14:00

def predict_wma(series, window=7):
    # Create weights (more recent days have higher weight)
    weights = np.arange(1, window + 1)
    weights = weights / weights.sum()
    
    # We want to average the same hour over the last 'window' days.
    # So we shift the series by 24 hours, 48 hours, etc.
    shifted_series = [series.shift(24 * i) for i in range(1, window + 1)]
    
    # Multiply by weights and sum
    prediction = sum(s * w for s, w in zip(shifted_series, weights))
    return prediction

df['pred_wma_consumption'] = predict_wma(df['actual_consumption'], window=7)
df['pred_wma_wind'] = predict_wma(df['actual_wind'], window=7)
df['pred_wma_solar'] = predict_wma(df['actual_solar'], window=7)

df.tail()

In [ ]:
# Evaluation Metrics
def wape(y_true, y_pred):
    mask = ~y_true.isna() & ~y_pred.isna()
    return (np.abs(y_true[mask] - y_pred[mask]).sum() / y_true[mask].sum()) * 100

metrics = {
    'Consumption - EPİAŞ (Yük Tahmini) WAPE (%)': wape(df['actual_consumption'], df['epias_load_forecast']),
    'Consumption - Our WMA Baseline WAPE (%)': wape(df['actual_consumption'], df['pred_wma_consumption']),
    
    'Wind - EPİAŞ (KGÜP) WAPE (%)': wape(df['actual_wind'], df['epias_kgup_wind']),
    'Wind - Our WMA Baseline WAPE (%)': wape(df['actual_wind'], df['pred_wma_wind']),
    
    'Solar - EPİAŞ (KGÜP) WAPE (%)': wape(df['actual_solar'], df['epias_kgup_solar']),
    'Solar - Our WMA Baseline WAPE (%)': wape(df['actual_solar'], df['pred_wma_solar']),
}

for k, v in metrics.items():
    print(f"{k}: {v:.2f}%")

In [ ]:
# Plotting
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Last 7 days
plot_df = df.tail(24 * 7)

axes[0].plot(plot_df.index, plot_df['actual_consumption'], label='Actual Consumption', color='black', linewidth=2)
axes[0].plot(plot_df.index, plot_df['epias_load_forecast'], label='EPİAŞ Forecast', linestyle='--')
axes[0].plot(plot_df.index, plot_df['pred_wma_consumption'], label='Our WMA Baseline', linestyle='-.')
axes[0].set_title('Consumption: Actual vs EPİAŞ vs Baseline')
axes[0].legend()

axes[1].plot(plot_df.index, plot_df['actual_wind'], label='Actual Wind Gen', color='black', linewidth=2)
axes[1].plot(plot_df.index, plot_df['epias_kgup_wind'], label='EPİAŞ KGUP Wind', linestyle='--')
axes[1].plot(plot_df.index, plot_df['pred_wma_wind'], label='Our WMA Baseline', linestyle='-.')
axes[1].set_title('Wind Generation: Actual vs EPİAŞ vs Baseline')
axes[1].legend()

plt.tight_layout()
plt.show()